In [ ]:
import yaml
from pathlib import Path

_REPO_ROOT = Path("../../..")
with open(_REPO_ROOT / "configs/data_paths_PPI.yaml") as _f:
    _paths = yaml.safe_load(_f)
SAR_ROOT = _paths["SAR_sea_ice_dataset"]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
from pathlib import Path
import sys

project_root = _REPO_ROOT.resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.sea_ice_drift.adapted_dualPol import warp_with_forward_flow

In [ ]:
# pixel displacement
past_file_d = f'{SAR_ROOT}/VECTOR_FIELDS_24h_pairs/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/07/20141106T0905__20141107T0807_past.npz'
future_file_d = f'{SAR_ROOT}/VECTOR_FIELDS_24h_pairs/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/07/20141107T0807__20141108T0848_future.npz'

In [ ]:
# velocity
past_file_v = f'{SAR_ROOT}/VECTOR_FIELDS_24h_pairs_velocity/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/07/20141106T0905__20141107T0807_past.npz'
future_file_v = f'{SAR_ROOT}/VECTOR_FIELDS_24h_pairs_velocity/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/07/20141107T0807__20141108T0848_future.npz'

In [ ]:
from pathlib import Path

# path = Path(f"{SAR_ROOT}/VECTOR_FIELDS_24h_pairs/HV_HH/region-27_0-82_951-35_52-83_8")
path = Path(f"{SAR_ROOT}/VECTOR_FIELDS_24h_pairs/HV_HH/region-33_0-82_95-40_715-83_903")



files = sorted(path.rglob("*.npz"))

for f in files:
    print(f.relative_to(path))



In [ ]:
def plot_drift(npz_file, quiver_step=80, save=None):
    """
    Visualize SAR drift from one NPZ vector field.
    
    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Start image + vectors
       [4] Warped(start) using drift
    """
    
    npz_file = Path(npz_file)

    # Load drift + metadata
    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item()

    u = vf["u"]
    v = vf["v"]
    # print(u[500, 500], v[500, 500])

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting drift:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")

    # Load SAR images (HV band index 1)
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[1].values.astype(float)
    img2 = da2[1].values.astype(float)

    # Warp image1 with the drift field
    # img1_warp = warp_image_with_flow(img1, u, v)
    img1_warp = warp_with_forward_flow(img1, u, v)
    # img1_warp = warp_image_with_flow_anchor(img1, u, v, anchor_dx=50, anchor_dy=-50)


    # Prepare quiver grid
    H, W = u.shape
    yy = np.arange(0, H, quiver_step)
    xx = np.arange(0, W, quiver_step)
    Xq, Yq = np.meshgrid(xx, yy)
    Uq = u[yy[:,None], xx]
    Vq = v[yy[:,None], xx]

    # ---- Plot figures ---- #
    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    ax_start, ax_end, ax_vec, ax_warp = axes.flatten()

    # 1 — Start image in dB
    ax_start.imshow(10*np.log10(img1), cmap="gray")
    ax_start.set_title(f"Start image (t0): {start_tiff.stem}")

    
    # 2 — End image in dB
    ax_end.imshow(10*np.log10(img2), cmap="gray")
    ax_end.set_title(f"End image (t1): {end_tiff.stem}")

    # 3 — Vector field on start image
    ax_vec.imshow(10*np.log10(img1), cmap="gray")
    ax_vec.quiver(
        Xq, Yq, Uq, Vq,
        color="red",
        scale_units="xy",
        scale=1, # need to change to 0.001 or so if plotting velocities
        angles="xy",
        width=0.003
    )
    ax_vec.set_title("Start image + vector field")

    # 4 — Warped start image
    ax_warp.imshow(10*np.log10(img1_warp), cmap="gray")
    ax_warp.set_title("Warped start image using drift field")

    # Remove axis ticks
    for ax in axes.flatten():
        # ax.set_xticks([])
        # ax.set_yticks([])
        ax.grid(color="white", alpha=1, linewidth=0.5)

    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=300)
    plt.show()


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr

def plot_drift_divergence(npz_file, pixel_size_m=100.0, cmap="RdBu_r", clip_percentile=99):
    """
    Visualize SAR drift divergence from one NPZ vector field.

    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Start image + divergence (1/s)
       [4] Warped(start) using drift

    Notes:
      - If u,v are m/s, divergence returned is 1/s when pixel_size_m is in meters.
      - If your u,v are pixels (displacements), divergence will not be physical unless you convert first.
    """
    npz_file = Path(npz_file)

    # Load drift + metadata
    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item() if "meta" in vf else {}

    u = vf["u"].astype(np.float32, copy=False)
    v = vf["v"].astype(np.float32, copy=False)

    # Prefer pixel size from meta if available
    px_m = float(meta.get("pixel_size_m", pixel_size_m))

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting divergence:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")
    print(f"  pixel_size_m: {px_m}")

    # Load SAR images
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)
    img1 = da1[0].values.astype(float)
    img2 = da2[0].values.astype(float)

    # Warp image1 with the drift field (same as your existing function)
    # img1_warp = warp_image_with_flow(img1, u, v)
    img1_warp = warp_with_forward_flow(img1, u, v)

    # ---- Divergence (1/s) ----
    # np.gradient returns d/d(row), d/d(col) if you pass a 2D array.
    # So: du/dx = du/dcol / dx_m, dv/dy = dv/drow / dy_m
    # du_drow, du_dcol = np.gradient(u)
    # dv_drow, dv_dcol = np.gradient(v)

    # div = (du_dcol + dv_drow) / px_m  # 1/s if u,v are m/s and px_m in m
######################
    from scipy.ndimage import gaussian_filter

    sigma_px = 10  # try 1–3 pixels

    u_s = gaussian_filter(u, sigma=sigma_px, mode="nearest")
    v_s = gaussian_filter(v, sigma=sigma_px, mode="nearest")

    du_drow, du_dcol = np.gradient(u_s)
    dv_drow, dv_dcol = np.gradient(v_s)

    div = (du_dcol + dv_drow) / px_m
###################

    # Robust color limits
    finite = np.isfinite(div)
    if np.any(finite):
        vmax = np.nanpercentile(np.abs(div[finite]), clip_percentile)
        if vmax == 0 or not np.isfinite(vmax):
            vmax = np.nanmax(np.abs(div[finite])) if np.nanmax(np.abs(div[finite])) > 0 else 1.0
    else:
        vmax = 1.0

    vmin = -vmax

    # ---- Plot figures ---- #
    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    ax_start, ax_end, ax_div, ax_warp = axes.flatten()

    ax_start.imshow(10*np.log10(img1), cmap="gray")
    ax_start.set_title(f"Start image (t0): {start_tiff.stem}")

    ax_end.imshow(10*np.log10(img2), cmap="gray")
    ax_end.set_title(f"End image (t1): {end_tiff.stem}")

    # Divergence overlay
    # im = ax_div.imshow(div, cmap=cmap, vmin=vmin, vmax=vmax, alpha=1)
    # ax_div.imshow(10*np.log10(img1), cmap="gray", alpha=0.3)

    # ax_div.set_title("Divergence overlay")
    # cbar = fig.colorbar(im, ax=ax_div, fraction=0.046, pad=0.04)
    # cbar.set_label("1/s")
    # Normalize divergence magnitude to [0, 1]
    abs_div = np.abs(div)

    # Use same vmax as colormap
    alpha = abs_div / vmax
    alpha = np.clip(alpha, 0, 1)

    # Optional: nonlinear boost to emphasize strong features
    alpha = alpha**0.8   # try 0.5–1.0

    # Plot
    ax_div.imshow(10*np.log10(img1), cmap="gray", alpha=1.0)
    im = ax_div.imshow(div, cmap=cmap, vmin=vmin, vmax=vmax, alpha=alpha)

    ax_div.set_title("Divergence overlay")
    cbar = fig.colorbar(im, ax=ax_div, fraction=0.046, pad=0.04)
    cbar.set_label("1/s")


    ax_warp.imshow(10*np.log10(img1_warp), cmap="gray")
    ax_warp.set_title("Warped start image using drift field")

    for ax in axes.flatten():
        ax.grid(color="white", alpha=1, linewidth=0.5)

    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import rioxarray as rxr
from scipy.ndimage import map_coordinates
from matplotlib.ticker import FormatStrFormatter

# ---------- LaTeX / publication style ----------
TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

setup_pub_style(fontsize=9)

from datetime import datetime
import os

def fname_to_datetime_str(path):
    """
    Convert filename like 20190425T0735.tiff
    to '2019-04-25 07:35'
    """
    base = os.path.splitext(os.path.basename(path))[0]
    try:
        dt = datetime.strptime(base, "%Y%m%dT%H%M")
        return dt.strftime("%Y-%m-%d %H:%M")
    except Exception:
        return base  # fallback if unexpected format


def to_db(x, eps=1e-6):
    return 10 * np.log10(np.maximum(x.astype(np.float32), eps))

def extent_aspect_from_rioxarray(da):
    """
    Build (left, right, bottom, top) extent + aspect from a rioxarray DataArray.
    Assumes north-up georeferenced raster.
    """
    # bounds: left, bottom, right, top
    left, bottom, right, top = da.rio.bounds()

    # pixel sizes (meters or degrees depending on CRS)
    resx, resy = da.rio.resolution()
    aspect = abs(resx) / abs(resy)

    extent = (left, right, bottom, top)
    return extent, aspect

# def warp_with_forward_flow(img, u, v, n_iter=8, order=1, mode='constant', cval=np.nan):
#     rows, cols = img.shape
#     rr, cc = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')

#     r = rr.astype(np.float64)
#     c = cc.astype(np.float64)

#     for _ in range(n_iter):
#         v_rc = map_coordinates(v, [r, c], order=1, mode='nearest')
#         u_rc = map_coordinates(u, [r, c], order=1, mode='nearest')
#         r = rr - v_rc
#         c = cc - u_rc

#     warped = map_coordinates(img.astype(np.float64), [r, c], order=order, mode=mode, cval=cval)
#     return warped

def plot_drift(npz_file, quiver_step=80, height_ratio=0.70, pol="HV", invert_color=False, save=None):
    """
    Visualize SAR drift from one NPZ vector field with georeferenced lon/lat ticks.

    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Start image + vectors
       [4] Warped(start) using drift
    """
    if pol == "HV":
        pol_idx = 1
        polmin, polmax = -28, -14
    elif pol == "HH":
        pol_idx = 0
        polmin, polmax = -18, -8
    else:
        print('Invalid pol')
        exit()
    npz_file = Path(npz_file)

    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item()

    u = vf["u"]
    v = vf["v"]

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting drift:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")

    # Load SAR images (HV band index 1)
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[pol_idx].values.astype(float)
    img2 = da2[pol_idx].values.astype(float)

    # Georeferenced extent/aspect from tiff0 (assume both share CRS/grid)
    extent, aspect = extent_aspect_from_rioxarray(da1)

    # Warp start image with drift field (pixel flow)
    img1_warp = warp_with_forward_flow(img1, u, v)

    # Quiver grid in pixel coords
    H, W = u.shape
    yy = np.arange(0, H, quiver_step)
    xx = np.arange(0, W, quiver_step)
    Xq_pix, Yq_pix = np.meshgrid(xx, yy)
    Uq = u[yy[:, None], xx]
    Vq = v[yy[:, None], xx]

    # Convert quiver positions from pixel coords to map coords (extent space)
    left, right, bottom, top = extent
    xres = (right - left) / W
    yres = (top - bottom) / H  # positive if top>bottom

    Xq_map = left + Xq_pix * xres
    Yq_map = top  - Yq_pix * yres  # pixel row increases downward

    # Convert pixel displacements to map units for quiver arrows
    # u: +cols (x), v: +rows (down)
    Uq_map = Uq * xres
    Vq_map = -Vq * yres  # negative because rows down means y decreases

    # ---- Figure ----
    fig, axes = plt.subplots(
        2, 2,
        figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio),
        dpi=300
    )
    ax_start, ax_end, ax_vec, ax_warp = axes.flatten()

    # helper to apply consistent geo ticks
    def apply_geo_ticks(ax, show_ylabel=True, show_xlabel=True):
        ax.set_aspect(aspect)
        ax.grid(color="white", alpha=1, linewidth=0.5)

        # ticks
        xt = np.linspace(left, right, 5)
        yt = np.linspace(top, bottom, 5)  # top->bottom so labels match "north up"
        ax.set_xticks(xt)
        ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        if show_ylabel:
            ax.set_ylabel("Lat")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)
        
        if show_xlabel:
            ax.set_xlabel("Lon")
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

    plot_color = "gray"
    if invert_color:
        plot_color = "gray_r"
    # 1 — Start image
    ax_start.imshow(to_db(img1), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    # ax_start.set_title(f"Start image (t0): {start_tiff.stem}")
    start_dt = fname_to_datetime_str(start_tiff)
    ax_start.set_title(f"(a) $t_0$  {start_dt}")

    apply_geo_ticks(ax_start, show_ylabel=True, show_xlabel=False)

    # 2 — End image
    ax_end.imshow(to_db(img2), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    # ax_end.set_title(f"End image (t1): {end_tiff.stem}")
    end_dt   = fname_to_datetime_str(end_tiff)
    ax_end.set_title(f"(b) $t_1$  {end_dt}")

    apply_geo_ticks(ax_end, show_ylabel=False, show_xlabel=False)

    # 3 — Vector field on start image
    ax_vec.imshow(to_db(img1), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    ax_vec.quiver(
        Xq_map, Yq_map, Uq_map, Vq_map,
        color="#F0E442",
        angles="xy",
        scale_units="xy",
        scale=1.0,           # because U,V are now in map units
        width=0.009
    )
    ax_vec.set_title(r"(c) $\Delta \boldsymbol{x}_{t_0 \rightarrow t_1}$")
    apply_geo_ticks(ax_vec, show_ylabel=True)

    # 4 — Warped start image
    ax_warp.imshow(to_db(img1_warp), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    ax_warp.set_title(r"(d) $\mathbf{x}_{t_1} = \mathbf{x}_{t_0} + \Delta \mathbf{x}_{t_0\rightarrow t_1}$")
    apply_geo_ticks(ax_warp, show_ylabel=False)

    fig.tight_layout()
    if save:
        plt.savefig(save, dpi=300)
    plt.show()
    return fig, axes


In [ ]:
from scipy.ndimage import gaussian_filter

def plot_drift_divergence_two_npz(
    npz_vel,                 # u,v are VELOCITIES (e.g., m/s) -> used for divergence
    npz_disp,                # u,v are DISPLACEMENTS (pixels or meters over dt) -> used for warping
    pol="HV",
    invert_color=False,
    pixel_size_m=100.0,      # used for divergence scaling
    cmap="RdBu_r",
    clip_percentile=99,
    sigma_px=10.0,           # smoothing (in pixels) applied to velocity field before gradient
    alpha_gamma=0.8,         # alpha = (|div|/vmax)^gamma
    height_ratio=0.70,
    save=None,
):
    """
    2x2 figure:
      [1] Start image (dB)
      [2] End image (dB)
      [3] Divergence overlay (computed from velocity NPZ)
      [4] Warped start image (using displacement NPZ)

    Assumptions:
      - start/end paths are taken from npz_disp meta (fallback: npz_vel meta)
      - divergence uses u_vel, v_vel; warping uses u_disp, v_disp
      - divergence unit is 1/s if u_vel,v_vel are in m/s and pixel_size_m is meters
    """
    # --- polarization settings ---
    if pol == "HV":
        pol_idx = 1
        polmin, polmax = -28, -14
    elif pol == "HH":
        pol_idx = 0
        polmin, polmax = -18, -8
    else:
        raise ValueError("pol must be 'HV' or 'HH'")

    npz_vel = Path(npz_vel)
    npz_disp = Path(npz_disp)

    vf_vel  = np.load(npz_vel, allow_pickle=True)
    vf_disp = np.load(npz_disp, allow_pickle=True)

    meta_vel  = vf_vel["meta"].item()  if "meta" in vf_vel  else {}
    meta_disp = vf_disp["meta"].item() if "meta" in vf_disp else {}

    # Velocity for divergence
    u_vel = vf_vel["u"].astype(np.float32, copy=False)
    v_vel = vf_vel["v"].astype(np.float32, copy=False)

    # Displacement for warping
    u_disp = vf_disp["u"].astype(np.float32, copy=False)
    v_disp = vf_disp["v"].astype(np.float32, copy=False)

    # Prefer pixel size from velocity meta if available
    px_m = float(meta_vel.get("pixel_size_m", pixel_size_m))

    # Start/end paths: prefer displacement meta (since it's what you're warping with)
    start_path = meta_disp.get("start_path", meta_vel.get("start_path", None))
    end_path   = meta_disp.get("end_path",   meta_vel.get("end_path",   None))
    if start_path is None or end_path is None:
        raise KeyError("Could not find start_path/end_path in either NPZ meta.")

    start_tiff = Path(start_path)
    end_tiff   = Path(end_path)

    print("Plotting divergence + warping (two NPZ):")
    print(f"  vel_npz : {npz_vel}")
    print(f"  disp_npz: {npz_disp}")
    print(f"  Start image: {start_tiff}")
    print(f"  End image:   {end_tiff}")
    print(f"  pixel_size_m used for divergence: {px_m}")

    # --- Load SAR images ---
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[pol_idx].values.astype(float)
    img2 = da2[pol_idx].values.astype(float)

    extent, aspect = extent_aspect_from_rioxarray(da1)
    left, right, bottom, top = extent

    # --- Warp using displacement ---
    img1_warp = warp_with_forward_flow(img1, u_disp, v_disp)

    # --- Divergence from velocity (smooth -> gradient -> div) ---
    u_s = gaussian_filter(u_vel, sigma=sigma_px, mode="nearest")
    v_s = gaussian_filter(v_vel, sigma=sigma_px, mode="nearest")

    du_drow, du_dcol = np.gradient(u_s)
    dv_drow, dv_dcol = np.gradient(v_s)

    div = (du_dcol + dv_drow) / px_m

    # Robust symmetric color limits
    finite = np.isfinite(div)
    if np.any(finite):
        vmax = np.nanpercentile(np.abs(div[finite]), clip_percentile)
        if vmax == 0 or not np.isfinite(vmax):
            vmax = np.nanmax(np.abs(div[finite])) if np.nanmax(np.abs(div[finite])) > 0 else 1.0
    else:
        vmax = 1.0
    vmin = -vmax

    # Alpha based on |div|
    abs_div = np.abs(div)
    alpha = np.clip(abs_div / vmax, 0, 1) ** alpha_gamma

    # --- Plot ---
    plot_cmap = "gray_r" if invert_color else "gray"
    start_dt = fname_to_datetime_str(start_tiff)
    end_dt   = fname_to_datetime_str(end_tiff)

    fig, axes = plt.subplots(
        2, 2,
        figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio),
        dpi=300
    )
    ax_start, ax_end, ax_div, ax_warp = axes.flatten()

    def apply_geo_ticks(ax, show_ylabel=True, show_xlabel=True):
        ax.set_aspect(aspect)
        ax.grid(color="white", alpha=1, linewidth=0.5)

        xt = np.linspace(left, right, 5)
        yt = np.linspace(top, bottom, 5)
        ax.set_xticks(xt)
        ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        if show_ylabel:
            ax.set_ylabel("Lat")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

        if show_xlabel:
            ax.set_xlabel("Lon")
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

    # [1] Start
    ax_start.imshow(to_db(img1), cmap=plot_cmap, extent=extent, origin="upper",
                    vmin=polmin, vmax=polmax)
    ax_start.set_title(f"$t_0$  {start_dt}")
    apply_geo_ticks(ax_start, show_ylabel=True, show_xlabel=False)

    # [2] End
    ax_end.imshow(to_db(img2), cmap=plot_cmap, extent=extent, origin="upper",
                  vmin=polmin, vmax=polmax)
    ax_end.set_title(f"$t_1$  {end_dt}")
    apply_geo_ticks(ax_end, show_ylabel=False, show_xlabel=False)

    # [3] Divergence overlay
    ax_div.imshow(to_db(img1), cmap=plot_cmap, extent=extent, origin="upper",
                  vmin=polmin, vmax=polmax, alpha=1.0)
    im = ax_div.imshow(div, cmap=cmap, vmin=vmin, vmax=vmax,
                       extent=extent, origin="upper", alpha=alpha)
    ax_div.set_title(r"$\nabla\!\cdot\boldsymbol{u}_{t_0 \rightarrow t_1}$")
    apply_geo_ticks(ax_div, show_ylabel=True, show_xlabel=True)

    # Colorbar that doesn't squash the subplot: put it in a fixed axis
    fig.subplots_adjust(right=0.88)
    cax = fig.add_axes([0.495, 0.095, 0.015, 0.39])  # [left, bottom, width, height]
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label("1/s")
    cbar.ax.yaxis.set_label_coords(3, 0.5)

    # [4] Warped
    ax_warp.imshow(to_db(img1_warp), cmap=plot_cmap, extent=extent, origin="upper",
                   vmin=polmin, vmax=polmax)
    ax_warp.set_title(r"$\mathbf{x}_{t_1} = \mathbf{x}_{t_0} + \Delta \mathbf{x}_{t_0\rightarrow t_1}$")
    apply_geo_ticks(ax_warp, show_ylabel=False, show_xlabel=True)

    fig.tight_layout()

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=300)

    plt.show()
    return fig, axes


In [ ]:
# plot_drift(future_file_d, quiver_step=80, height_ratio=1, pol="HV")
fig, _ = plot_drift_divergence_two_npz(npz_vel = future_file_v, npz_disp = future_file_d, height_ratio=.95, cmap='coolwarm')
fig.savefig("lead_divergence.pdf", dpi=300)

In [ ]:
from matplotlib.tri import Triangulation
import numpy as np

def get_deformation_elems(x, y, u, v, a):
    """ Compute deformation for given elements.

    Input X, Y, U, V are organized in three columns: for each node of M elements.
    To convert deformation rates from 1/s to %/day outputs should be multiplied by 8640000.

    Parameters
    ----------
    x : 3xM ndarray
        X-coordinates of nodes, m
    y : 3xM ndarray
        Y-coordinates of nodes, m
    u : 3xM ndarray
        U-component of nodes, m/s
    v : 3xM ndarray
        V-component of nodes, m/s
    a : Mx1 ndarray
        area of elements, m2

    Returns
    -------
    e1 : Mx1 array
        Divergence, 1/s
    e2 : Mx1 array
        Shear, 1/s
    e3 : Mx1 array
        Vorticity, 1/s

    """
    # contour integrals of u and v [m/s * m ==> m2/s]
    ux = uy = vx = vy = 0
    for i0, i1 in zip([1, 2, 0], [0, 1, 2]):
        ux += (u[i0] + u[i1]) * (y[i0] - y[i1])
        uy -= (u[i0] + u[i1]) * (x[i0] - x[i1])
        vx += (v[i0] + v[i1]) * (y[i0] - y[i1])
        vy -= (v[i0] + v[i1]) * (x[i0] - x[i1])
    # divide integral by double area [m2/s / m2 ==> 1/day]
    ux, uy, vx, vy =  [i / (2 * a) for i in (ux, uy, vx, vy)]

    # deformation components
    e1 = ux + vy
    e2 = ((ux - vy) ** 2 + (uy + vx) ** 2) ** 0.5
    e3 = vx - uy

    return e1, e2, e3

def get_deformation_on_triangulation(x, y, u, v, t):
    """ Compute deformation for given nodes.

    Input X, Y, U, V are given for individual N nodes. Nodes coordinates are triangulated and
    area, perimeter and deformation is computed for M elements.

    Parameters
    ----------
    x : Nx1 ndarray
        X-coordinates of nodes, m
    y : Nx1 ndarray
        Y-coordinates of nodes, m
    u : Nx1 ndarray
        U-component of nodes, m/s
    v : Nx1 ndarray
        V-component of nodes, m/s
    t : 3xM array
        Triangulation (indices of input nodes for each element)

    Returns
    -------
    e1 : Mx1 array
        Divergence, 1/s
    e2 : Mx1 array
        Shear, 1/s
    e3 : Mx1 array
        Vorticity, 1/s
    a : Mx1 array
        Area, m2
    p : Mx1 array
        Perimeter, m
    """

    # coordinates and speeds of corners of each element
    xt, yt, ut, vt = [i[t].T for i in (x, y, u, v)]

    # side lengths (X,Y,tot)
    tri_x = np.diff(np.vstack([xt, xt[0]]), axis=0)
    tri_y = np.diff(np.vstack([yt, yt[0]]), axis=0)
    tri_s = np.hypot(tri_x, tri_y)
    # perimeter
    tri_p = np.sum(tri_s, axis=0)
    s = tri_p/2
    # area
    tri_a = np.sqrt(s * (s - tri_s[0]) * (s - tri_s[1]) * (s - tri_s[2]))

    # deformation components
    e1, e2, e3 = get_deformation_elems(xt, yt, ut, vt, tri_a)

    return e1, e2, e3, tri_a, tri_p

def get_deformation_nodes(x, y, u, v):
    """ Compute deformation for given nodes.

    Input X, Y, U, V are given for individual N nodes. Nodes coordinates are triangulated and
    area, perimeter and deformation is computed for M elements.

    Parameters
    ----------
    x : Nx1 ndarray
        X-coordinates of nodes, m
    y : Nx1 ndarray
        Y-coordinates of nodes, m
    u : Nx1 ndarray
        U-component of nodes, m/s
    v : Nx1 ndarray
        V-component of nodes, m/s

    Returns
    -------
    e1 : Mx1 array
        Divergence, 1/s
    e2 : Mx1 array
        Shear, 1/s
    e3 : Mx1 array
        Vorticity, 1/s
    a : Mx1 array
        Area, m2
    p : Mx1 array
        Perimeter, m
    t : 3xM array
        Triangulation (indices of input nodes for each element)
    """
    tri = Triangulation(x, y)

    e1, e2, e3, tri_a, tri_p = get_deformation_on_triangulation(x, y, u, v, tri.triangles)

    return e1, e2, e3, tri_a, tri_p, tri.triangles

In [ ]:
def plot_drift_deformation_triangles(
    npz_vel,
    npz_disp,
    pol="HV",
    field="div",                 # "div", "shear", or "vort"
    invert_color=False,
    pixel_size_m=100.0,
    stride=8,                    # subsample for triangulation
    unit="1/day",                # "1/s", "1/day", or "%/day"
    cmap="RdBu_r",
    clip_percentile=99,
    alpha=0.85,
    height_ratio=0.70,
    save=None,
):
    """
    2x2 figure:
      [1] Start image
      [2] End image
      [3] Triangulation-based deformation overlay
      [4] Warped start image

    field:
      - "div"   -> divergence
      - "shear" -> shear
      - "vort"  -> vorticity
    """

    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    import rioxarray as rxr
    from matplotlib.tri import Triangulation
    from matplotlib.ticker import FormatStrFormatter

    # --- polarization settings ---
    if pol == "HV":
        pol_idx = 1
        polmin, polmax = -28, -14
    elif pol == "HH":
        pol_idx = 0
        polmin, polmax = -18, -8
    else:
        raise ValueError("pol must be 'HV' or 'HH'")

    npz_vel = Path(npz_vel)
    npz_disp = Path(npz_disp)

    vf_vel  = np.load(npz_vel, allow_pickle=True)
    vf_disp = np.load(npz_disp, allow_pickle=True)

    meta_vel  = vf_vel["meta"].item()  if "meta" in vf_vel  else {}
    meta_disp = vf_disp["meta"].item() if "meta" in vf_disp else {}

    u_vel = vf_vel["u"].astype(np.float32, copy=False)
    v_vel = vf_vel["v"].astype(np.float32, copy=False)

    u_disp = vf_disp["u"].astype(np.float32, copy=False)
    v_disp = vf_disp["v"].astype(np.float32, copy=False)

    px_m = float(meta_vel.get("pixel_size_m", pixel_size_m))

    start_path = meta_disp.get("start_path", meta_vel.get("start_path", None))
    end_path   = meta_disp.get("end_path",   meta_vel.get("end_path",   None))
    if start_path is None or end_path is None:
        raise KeyError("Could not find start_path/end_path in either NPZ meta.")

    start_tiff = Path(start_path)
    end_tiff   = Path(end_path)

    print("Plotting triangulation-based deformation:")
    print(f"  vel_npz : {npz_vel}")
    print(f"  disp_npz: {npz_disp}")
    print(f"  field   : {field}")
    print(f"  stride  : {stride}")
    print(f"  unit    : {unit}")
    print(f"  pixel_size_m: {px_m}")

    # --- Load SAR images ---
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[pol_idx].values.astype(float)
    img2 = da2[pol_idx].values.astype(float)

    extent, aspect = extent_aspect_from_rioxarray(da1)
    left, right, bottom, top = extent

    # --- Warp using displacement ---
    img1_warp = warp_with_forward_flow(img1, u_disp, v_disp)

    # --- Build node coordinates on image grid in meters ---
    nrows, ncols = u_vel.shape

    rows = np.arange(0, nrows, stride)
    cols = np.arange(0, ncols, stride)
    cc, rr = np.meshgrid(cols, rows)

    # Coordinates in meters relative to image grid
    # Using pixel-center coordinates
    x_nodes = cc.astype(np.float64) * px_m
    y_nodes = rr.astype(np.float64) * px_m

    u_nodes = u_vel[rows[:, None], cols[None, :]]
    v_nodes = v_vel[rows[:, None], cols[None, :]]

    x_flat = x_nodes.ravel()
    y_flat = y_nodes.ravel()
    u_flat = u_nodes.ravel()
    v_flat = v_nodes.ravel()

    # Remove invalid nodes
    good = np.isfinite(x_flat) & np.isfinite(y_flat) & np.isfinite(u_flat) & np.isfinite(v_flat)
    x_flat = x_flat[good]
    y_flat = y_flat[good]
    u_flat = u_flat[good]
    v_flat = v_flat[good]

    # --- Compute deformation on triangulation ---
    e1, e2, e3, tri_a, tri_p, triangles = get_deformation_nodes(x_flat, y_flat, u_flat, v_flat)

    if field == "div":
        values = e1
        title = r"$\nabla\!\cdot\mathbf{u}$"
    elif field == "shear":
        values = e2
        title = r"Shear"
    elif field == "vort":
        values = e3
        title = r"Vorticity"
    else:
        raise ValueError("field must be 'div', 'shear', or 'vort'")

    # --- Unit conversion ---
    if unit == "1/s":
        scale = 1.0
        cbar_label = "1/s"
    elif unit == "1/day":
        scale = 86400.0
        cbar_label = "1/day"
    elif unit == "%/day":
        scale = 8640000.0
        cbar_label = "%/day"
    else:
        raise ValueError("unit must be '1/s', '1/day', or '%/day'")

    values = values * scale

    # triangle centers for masking / robust limits
    finite = np.isfinite(values)
    if np.any(finite):
        vmax = np.nanpercentile(np.abs(values[finite]), clip_percentile)
        if vmax == 0 or not np.isfinite(vmax):
            vmax = np.nanmax(np.abs(values[finite]))
            if vmax == 0 or not np.isfinite(vmax):
                vmax = 1.0
    else:
        vmax = 1.0

    # symmetric for div/vort, nonnegative for shear
    if field == "shear":
        vmin = 0.0
    else:
        vmin = -vmax

    # --- Convert node coords to map coords for plotting ---
    # Use same image extent as background
    # cols span [0, ncols-1], rows span [0, nrows-1]
    lon_nodes = left + (x_flat / px_m) * (right - left) / max(ncols - 1, 1)
    lat_nodes = top + (y_flat / px_m) * (bottom - top) / max(nrows - 1, 1)

    tri_plot = Triangulation(lon_nodes, lat_nodes, triangles=triangles)

    # --- Plot ---
    plot_cmap = "gray_r" if invert_color else "gray"
    start_dt = fname_to_datetime_str(start_tiff)
    end_dt   = fname_to_datetime_str(end_tiff)

    fig, axes = plt.subplots(
        2, 2,
        figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio),
        dpi=300
    )
    ax_start, ax_end, ax_def, ax_warp = axes.flatten()

    def apply_geo_ticks(ax, show_ylabel=True, show_xlabel=True):
        ax.set_aspect(aspect)
        ax.grid(color="white", alpha=1, linewidth=0.5)

        xt = np.linspace(left, right, 5)
        yt = np.linspace(top, bottom, 5)
        ax.set_xticks(xt)
        ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        if show_ylabel:
            ax.set_ylabel("Lat")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

        if show_xlabel:
            ax.set_xlabel("Lon")
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

    # [1] Start
    ax_start.imshow(to_db(img1), cmap=plot_cmap, extent=extent, origin="upper",
                    vmin=polmin, vmax=polmax)
    ax_start.set_title(f"(a) $t_0$  {start_dt}")
    apply_geo_ticks(ax_start, show_ylabel=True, show_xlabel=False)

    # [2] End
    ax_end.imshow(to_db(img2), cmap=plot_cmap, extent=extent, origin="upper",
                  vmin=polmin, vmax=polmax)
    ax_end.set_title(f"(b) $t_1$  {end_dt}")
    apply_geo_ticks(ax_end, show_ylabel=False, show_xlabel=False)

    # [3] Deformation overlay
    ax_def.imshow(to_db(img1), cmap=plot_cmap, extent=extent, origin="upper",
                  vmin=polmin, vmax=polmax, alpha=1.0)

    im = ax_def.tripcolor(
        tri_plot,
        values,
        shading="flat",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=alpha,
    )
    ax_def.set_title(r"(c) $\nabla\!\cdot\boldsymbol{u}_{t_0 \rightarrow t_1}$")
    apply_geo_ticks(ax_def, show_ylabel=True, show_xlabel=True)

    # colorbar
    fig.subplots_adjust(right=0.88)
    cax = fig.add_axes([0.47, 0.095, 0.015, 0.39])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(cbar_label)
    cbar.ax.yaxis.set_label_coords(5.5, 0.5)

    # [4] Warped
    ax_warp.imshow(to_db(img1_warp), cmap=plot_cmap, extent=extent, origin="upper",
                   vmin=polmin, vmax=polmax)
    ax_warp.set_title(r"(d) $\mathbf{x}_{t_1} = \mathbf{x}_{t_0} + \Delta \mathbf{x}_{t_0\rightarrow t_1}$")
    apply_geo_ticks(ax_warp, show_ylabel=False, show_xlabel=True)

    fig.tight_layout()

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=300)

    plt.show()
    return fig, axes

In [ ]:
def plot_drift_deformation_triangles(
    npz_vel,
    npz_disp,
    pol="HV",
    field="div",                 # "div", "shear", or "vort"
    invert_color=False,
    pixel_size_m=100.0,
    stride=8,                    # subsample for triangulation
    unit="1/day",                # "1/s", "1/day", or "%/day"
    cmap="RdBu_r",
    clip_percentile=99,
    alpha=0.85,
    height_ratio=0.70,
    save=None,
):
    """
    2x2 figure:
      [1] Start image
      [2] End image
      [3] Triangulation-based deformation overlay
      [4] Warped start image

    field:
      - "div"   -> divergence
      - "shear" -> shear
      - "vort"  -> vorticity
    """

    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    import rioxarray as rxr
    from matplotlib.tri import Triangulation
    from matplotlib.ticker import FormatStrFormatter

    # --- polarization settings ---
    if pol == "HV":
        pol_idx = 1
        polmin, polmax = -28, -14
    elif pol == "HH":
        pol_idx = 0
        polmin, polmax = -18, -8
    else:
        raise ValueError("pol must be 'HV' or 'HH'")

    npz_vel = Path(npz_vel)
    npz_disp = Path(npz_disp)

    vf_vel  = np.load(npz_vel, allow_pickle=True)
    vf_disp = np.load(npz_disp, allow_pickle=True)

    meta_vel  = vf_vel["meta"].item()  if "meta" in vf_vel  else {}
    meta_disp = vf_disp["meta"].item() if "meta" in vf_disp else {}

    u_vel = vf_vel["u"].astype(np.float32, copy=False)
    v_vel = vf_vel["v"].astype(np.float32, copy=False)

    u_disp = vf_disp["u"].astype(np.float32, copy=False)
    v_disp = vf_disp["v"].astype(np.float32, copy=False)

    px_m = float(meta_vel.get("pixel_size_m", pixel_size_m))

    start_path = meta_disp.get("start_path", meta_vel.get("start_path", None))
    end_path   = meta_disp.get("end_path",   meta_vel.get("end_path",   None))
    if start_path is None or end_path is None:
        raise KeyError("Could not find start_path/end_path in either NPZ meta.")

    start_tiff = Path(start_path)
    end_tiff   = Path(end_path)

    print("Plotting triangulation-based deformation:")
    print(f"  vel_npz : {npz_vel}")
    print(f"  disp_npz: {npz_disp}")
    print(f"  field   : {field}")
    print(f"  stride  : {stride}")
    print(f"  unit    : {unit}")
    print(f"  pixel_size_m: {px_m}")

    # --- Load SAR images ---
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[pol_idx].values.astype(float)
    img2 = da2[pol_idx].values.astype(float)

    extent, aspect = extent_aspect_from_rioxarray(da1)
    left, right, bottom, top = extent

    # --- Warp using displacement ---
    img1_warp = warp_with_forward_flow(img1, u_disp, v_disp)

    # --- Build node coordinates on image grid in meters ---
    nrows, ncols = u_vel.shape

    rows = np.arange(0, nrows, stride)
    cols = np.arange(0, ncols, stride)
    cc, rr = np.meshgrid(cols, rows)

    x_nodes = cc.astype(np.float64) * px_m
    y_nodes = rr.astype(np.float64) * px_m

    u_nodes = u_vel[rows[:, None], cols[None, :]]
    v_nodes = v_vel[rows[:, None], cols[None, :]]

    x_flat = x_nodes.ravel()
    y_flat = y_nodes.ravel()
    u_flat = u_nodes.ravel()
    v_flat = v_nodes.ravel()

    # Remove invalid nodes
    good = np.isfinite(x_flat) & np.isfinite(y_flat) & np.isfinite(u_flat) & np.isfinite(v_flat)
    x_flat = x_flat[good]
    y_flat = y_flat[good]
    u_flat = u_flat[good]
    v_flat = v_flat[good]

    # --- Compute deformation on triangulation ---
    e1, e2, e3, tri_a, tri_p, triangles = get_deformation_nodes(x_flat, y_flat, u_flat, v_flat)

    if field == "div":
        values = e1
    elif field == "shear":
        values = e2
    elif field == "vort":
        values = e3
    else:
        raise ValueError("field must be 'div', 'shear', or 'vort'")

    # --- Unit conversion ---
    if unit == "1/s":
        scale = 1.0
        cbar_label = "1/s"
    elif unit == "1/day":
        scale = 86400.0
        cbar_label = "1/day"
    elif unit == "%/day":
        scale = 8640000.0
        cbar_label = "%/day"
    else:
        raise ValueError("unit must be '1/s', '1/day', or '%/day'")

    values = values * scale

    finite = np.isfinite(values)
    if np.any(finite):
        vmax = np.nanpercentile(np.abs(values[finite]), clip_percentile)
        if vmax == 0 or not np.isfinite(vmax):
            vmax = np.nanmax(np.abs(values[finite]))
            if vmax == 0 or not np.isfinite(vmax):
                vmax = 1.0
    else:
        vmax = 1.0

    if field == "shear":
        vmin = 0.0
    else:
        vmin = -vmax

    # --- Convert node coords to map coords for plotting ---
    lon_nodes = left + (x_flat / px_m) * (right - left) / max(ncols - 1, 1)
    lat_nodes = top + (y_flat / px_m) * (bottom - top) / max(nrows - 1, 1)

    tri_plot = Triangulation(lon_nodes, lat_nodes, triangles=triangles)

    # --- Plot ---
    plot_cmap = "gray_r" if invert_color else "gray"
    start_dt = fname_to_datetime_str(start_tiff)
    end_dt   = fname_to_datetime_str(end_tiff)

    fig, axes = plt.subplots(
        2, 2,
        figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio),
        dpi=300
    )
    ax_start, ax_end, ax_def, ax_warp = axes.flatten()

    def apply_geo_ticks(ax, show_ylabel=True, show_xlabel=True):
        ax.set_aspect(aspect)
        ax.grid(color="white", alpha=1, linewidth=0.5)

        xt = np.linspace(left, right, 5)
        yt = np.linspace(top, bottom, 5)
        ax.set_xticks(xt)
        ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        if show_ylabel:
            ax.set_ylabel("Lat")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

        if show_xlabel:
            ax.set_xlabel("Lon")
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

    # [1] Start
    ax_start.imshow(to_db(img1), cmap=plot_cmap, extent=extent, origin="upper",
                    vmin=polmin, vmax=polmax)
    ax_start.set_title(f"(a) $t_0$  {start_dt}")
    apply_geo_ticks(ax_start, show_ylabel=True, show_xlabel=False)

    # [2] End
    ax_end.imshow(to_db(img2), cmap=plot_cmap, extent=extent, origin="upper",
                  vmin=polmin, vmax=polmax)
    ax_end.set_title(f"(b) $t_1$  {end_dt}")
    apply_geo_ticks(ax_end, show_ylabel=False, show_xlabel=False)

    # [3] Deformation overlay
    ax_def.imshow(to_db(img1), cmap=plot_cmap, extent=extent, origin="upper",
                  vmin=polmin, vmax=polmax, alpha=1.0)

    im = ax_def.tripcolor(
        tri_plot,
        values,
        shading="flat",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        alpha=alpha,
    )
    ax_def.set_title(r"(c) $\nabla\!\cdot\boldsymbol{u}_{t_0 \rightarrow t_1}$")
    apply_geo_ticks(ax_def, show_ylabel=True, show_xlabel=True)

    # [4] Warped
    ax_warp.imshow(to_db(img1_warp), cmap=plot_cmap, extent=extent, origin="upper",
                   vmin=polmin, vmax=polmax)
    ax_warp.set_title(r"(d) $\mathbf{x}_{t_0} + \Delta \mathbf{x}_{t_0\rightarrow t_1}$")
    apply_geo_ticks(ax_warp, show_ylabel=False, show_xlabel=True)

    # --- Layout: reserve space on both sides for colorbars ---
    # rect=[left, bottom, right, top] in figure coordinates
    fig.tight_layout(rect=[0, 0, 0.88, 1])

    # Read axis positions after layout is resolved
    pos_def  = ax_def.get_position()   # (c) bottom-left
    pos_warp = ax_warp.get_position()  # (d) bottom-right
    pos_end  = ax_end.get_position()   # (b) top-right

    cbar_w = 0.015
    gap    = 0.008

    # Deformation colorbar: just right of left column, aligned with bottom row (c)
    cax_def = fig.add_axes([
        pos_def.x1 + gap,
        pos_def.y0,
        cbar_w,
        pos_def.height,
    ])
    cbar = fig.colorbar(im, cax=cax_def)
    cbar.set_label(cbar_label, labelpad=-7.5)

    # SAR backscatter colorbar: right of right column, spanning top of (b) to bottom of (d)
    norm_sar = mcolors.Normalize(vmin=polmin, vmax=polmax)
    sm_sar = cm.ScalarMappable(cmap=plot_cmap, norm=norm_sar)
    sm_sar.set_array([])

    cax_sar = fig.add_axes([
        pos_warp.x1 + gap,
        pos_warp.y0,
        cbar_w,
        pos_end.y1 - pos_warp.y0,
    ])
    cbar_sar = fig.colorbar(sm_sar, cax=cax_sar)
    cbar_sar.set_label("HV backscatter [dB]")

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=300)

    plt.show()
    return fig, axes

In [ ]:
plot_drift_deformation_triangles(
    npz_vel=future_file_v,
    npz_disp=future_file_d,
    field="div",
    unit="1/day",
    stride=8,
    height_ratio=0.78,
    save="lead_divergence_triangles_ratio078.pdf",
)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import rioxarray as rxr
from scipy.ndimage import map_coordinates
from matplotlib.ticker import FormatStrFormatter

# ---------- LaTeX / publication style ----------
TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

setup_pub_style(fontsize=9)

from datetime import datetime
import os

def fname_to_datetime_str(path):
    """
    Convert filename like 20190425T0735.tiff
    to '2019-04-25 07:35'
    """
    base = os.path.splitext(os.path.basename(path))[0]
    try:
        dt = datetime.strptime(base, "%Y%m%dT%H%M")
        return dt.strftime("%Y-%m-%d %H:%M")
    except Exception:
        return base  # fallback if unexpected format


def to_db(x, eps=1e-6):
    return 10 * np.log10(np.maximum(x.astype(np.float32), eps))

def extent_aspect_from_rioxarray(da):
    """
    Build (left, right, bottom, top) extent + aspect from a rioxarray DataArray.
    Assumes north-up georeferenced raster.
    """
    # bounds: left, bottom, right, top
    left, bottom, right, top = da.rio.bounds()

    # pixel sizes (meters or degrees depending on CRS)
    resx, resy = da.rio.resolution()
    aspect = abs(resx) / abs(resy)

    extent = (left, right, bottom, top)
    return extent, aspect

# def warp_with_forward_flow(img, u, v, n_iter=8, order=1, mode='constant', cval=np.nan):
#     rows, cols = img.shape
#     rr, cc = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')

#     r = rr.astype(np.float64)
#     c = cc.astype(np.float64)

#     for _ in range(n_iter):
#         v_rc = map_coordinates(v, [r, c], order=1, mode='nearest')
#         u_rc = map_coordinates(u, [r, c], order=1, mode='nearest')
#         r = rr - v_rc
#         c = cc - u_rc

#     warped = map_coordinates(img.astype(np.float64), [r, c], order=order, mode=mode, cval=cval)
#     return warped

def plot_drift(npz_file, quiver_step=80, height_ratio=0.70, pol="HV", invert_color=False, save=None):
    """
    Visualize SAR drift from one NPZ vector field with georeferenced lon/lat ticks.

    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Start image + vectors
       [4] Warped(start) using drift
    """
    if pol == "HV":
        pol_idx = 1
        polmin, polmax = -28, -14
    elif pol == "HH":
        pol_idx = 0
        polmin, polmax = -18, -8
    else:
        print('Invalid pol')
        exit()
    npz_file = Path(npz_file)

    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item()

    u = vf["u"]
    v = vf["v"]

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting drift:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")

    # Load SAR images (HV band index 1)
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[pol_idx].values.astype(float)
    img2 = da2[pol_idx].values.astype(float)

    # Georeferenced extent/aspect from tiff0 (assume both share CRS/grid)
    extent, aspect = extent_aspect_from_rioxarray(da1)

    # Warp start image with drift field (pixel flow)
    img1_warp = warp_with_forward_flow(img1, u, v)

    # Quiver grid in pixel coords
    H, W = u.shape
    yy = np.arange(0, H, quiver_step)
    xx = np.arange(0, W, quiver_step)
    Xq_pix, Yq_pix = np.meshgrid(xx, yy)
    Uq = u[yy[:, None], xx]
    Vq = v[yy[:, None], xx]

    # Convert quiver positions from pixel coords to map coords (extent space)
    left, right, bottom, top = extent
    xres = (right - left) / W
    yres = (top - bottom) / H  # positive if top>bottom

    Xq_map = left + Xq_pix * xres
    Yq_map = top  - Yq_pix * yres  # pixel row increases downward

    # Convert pixel displacements to map units for quiver arrows
    # u: +cols (x), v: +rows (down)
    Uq_map = Uq * xres
    Vq_map = -Vq * yres  # negative because rows down means y decreases

    # ---- Figure ----
    fig, axes = plt.subplots(
        2, 2,
        figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio),
        dpi=300
    )
    ax_start, ax_end, ax_vec, ax_warp = axes.flatten()

    # helper to apply consistent geo ticks
    def apply_geo_ticks(ax, show_ylabel=True, show_xlabel=True):
        ax.set_aspect(aspect)
        ax.grid(color="white", alpha=1, linewidth=0.5)

        # ticks
        xt = np.linspace(left, right, 5)
        yt = np.linspace(top, bottom, 5)  # top->bottom so labels match "north up"
        ax.set_xticks(xt)
        ax.set_yticks(yt)
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        if show_ylabel:
            ax.set_ylabel("Lat")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)
        
        if show_xlabel:
            ax.set_xlabel("Lon")
        else:
            ax.set_xlabel("")
            ax.tick_params(labelbottom=False)

    plot_color = "gray"
    if invert_color:
        plot_color = "gray_r"

    # 1 — Start image
    ax_start.imshow(to_db(img1), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    start_dt = fname_to_datetime_str(start_tiff)
    ax_start.set_title(f"(a) $t_0$  {start_dt}")
    apply_geo_ticks(ax_start, show_ylabel=True, show_xlabel=False)

    # 2 — End image
    ax_end.imshow(to_db(img2), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    end_dt = fname_to_datetime_str(end_tiff)
    ax_end.set_title(f"(b) $t_1$  {end_dt}")
    apply_geo_ticks(ax_end, show_ylabel=False, show_xlabel=False)

    # 3 — Vector field on start image
    ax_vec.imshow(to_db(img1), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    ax_vec.quiver(
        Xq_map, Yq_map, Uq_map, Vq_map,
        color="#F0E442",
        angles="xy",
        scale_units="xy",
        scale=1.0,           # because U,V are now in map units
        width=0.009
    )
    ax_vec.set_title(r"(c) $\Delta \boldsymbol{x}_{t_0 \rightarrow t_1}$")
    apply_geo_ticks(ax_vec, show_ylabel=True)

    # 4 — Warped start image
    ax_warp.imshow(to_db(img1_warp), cmap=plot_color, extent=extent, origin="upper", vmin=polmin, vmax=polmax)
    ax_warp.set_title(r"(d) $\mathbf{x}_{t_0} + \Delta \mathbf{x}_{t_0\rightarrow t_1}$")
    apply_geo_ticks(ax_warp, show_ylabel=False)

    # Reserve space on the right for the SAR colorbar
    fig.tight_layout(rect=[0, 0, 0.88, 1])

    # Read axis positions after layout is resolved
    pos_end  = ax_end.get_position()   # (b) top-right
    pos_warp = ax_warp.get_position()  # (d) bottom-right

    cbar_w = 0.015
    gap    = 0.008

    # SAR backscatter colorbar: right of right column, spanning top of (b) to bottom of (d)
    norm_sar = mcolors.Normalize(vmin=polmin, vmax=polmax)
    sm_sar = cm.ScalarMappable(cmap=plot_color, norm=norm_sar)
    sm_sar.set_array([])

    cax_sar = fig.add_axes([
        pos_warp.x1 + gap,
        pos_warp.y0,
        cbar_w,
        pos_end.y1 - pos_warp.y0,
    ])
    cbar_sar = fig.colorbar(sm_sar, cax=cax_sar)
    cbar_sar.set_label("HH backscatter [dB]")

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=300)
    plt.show()
    return fig, axes

In [ ]:
i = 3207
future_file_d = str(files[i])
print(future_file_d)

In [ ]:
plot_drift(future_file_d, quiver_step=100, height_ratio=0.78, pol="HH", invert_color=True, save="floe_displacement_ratio078.pdf")